# 🌍 Mapa de Oportunidades y Simulador de Inversión
## Visualización Geoespacial & Decision Support System

**Objetivo:** Visualizar las desviaciones de valor detectadas por el modelo de ML en un mapa interactivo y crear una herramienta para simular escenarios de inversión basados en presupuesto y objetivos.

---

## 🧬 Contenido
1. [Carga de Datos e Integración Geoespacial](#1-carga)
2. [Mapa Interactivo de Valor (Heatmap de Desviación)](#2-mapa)
3. [Simulador de Inversión Estratégica](#3-simulador)
4. [Conclusiones Finales del Proyecto](#4-conclusiones)

In [ ]:
import pandas as pd
import folium
import json
from pathlib import Path
import branca.colormap as cm

# 1. CARGA DE DATOS ML Y GEOJSON
ml_path = Path("../data/barcelona_ml_valuation.csv")
geojson_path = Path("../data/raw/geojson/barrios_geojson_20251115_162533_398394.json")

df_ml = pd.read_csv(ml_path)
with open(geojson_path) as f:
    barrios_geo = json.load(f)

# Aseguramos que el barrio_id sea el índice para el mapeo rápido
df_ml['desviacion_valor'] = ((df_ml['avg_venta_23'] - df_ml['precio_estimado']) / df_ml['precio_estimado']) * 100

print(f"✅ {len(df_ml)} barrios cargados para mapeo.")
display(df_ml[['barrio_nombre', 'desviacion_valor']].sort_values('desviacion_valor').head(5))

## 2. Mapa Interactivo de Valor (¿Dónde están las gangas?) <a name="2-mapa"></a>

In [ ]:
# Inicializar el mapa centrado en Barcelona
m = folium.Map(location=[41.3851, 2.1734], zoom_start=12, tiles='cartodbpositron')

# Crear escala de colores: Verde (Infravalorado) -> Blanco (Ajustado) -> Rojo (Sobrevalorado)
colormap = cm.LinearColormap(
    colors=['#27ae60', '#f1c40f', '#e74c3c'],
    index=[-15, 0, 15],
    vmin=-20,
    vmax=20,
    caption='Desviación del Precio vs Fundamental ML (%)'
)

# Añadir Choropleth
folium.Choropleth(
    geo_data=barrios_geo,
    name='choropleth',
    data=df_ml,
    columns=['barrio_id', 'desviacion_valor'],
    key_on='feature.properties.codi_barri',
    fill_color='RdYlGn_r', # Red-Yellow-Green reversed para que verde sea negativo (mejor valor)
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name='Desviación de Precio ML (%)',
    highlight=True
).add_to(m)

# Añadir tooltips con información detallada
folium.GeoJson(
    barrios_geo,
    style_function=lambda x: {'fillColor': '#ffffff00', 'color': '#00000000'},
    tooltip=folium.GeoJsonTooltip(
        fields=['nom_barri'], 
        aliases=['Barrio:'],
        localize=True
    )
).add_to(m)

m.save('../data/barcelona_opportunity_map.html')
print("✅ Mapa interactivo generado: data/barcelona_opportunity_map.html")
m # Mostrar en el notebook

## 3. Simulador de Inversión Estratégica <a name="3-simulador"></a>

In [ ]:
def simulator(budget_euros, strategy='yield', min_size_m2=60):
    """
    Sugiere barrios basados en presupuesto y objetivo.
    strategy: 'yield' (mejor rentabilidad), 'safe' (menor desviación/riesgo), 'upside' (potencial crecimiento)
    """
    reco = df_ml.copy()
    
    # Calculamos precio total estimado de un piso de tamaño mínimo
    reco['entry_cost'] = reco['avg_venta_23'] * min_size_m2
    
    # Filtramos por presupuesto
    reco = reco[reco['entry_cost'] <= budget_euros]
    
    if reco.empty:
        return "❌ No hay opciones para este presupuesto en Barcelona actual."
    
    if strategy == 'yield':
        # Ordenar por rentabilidad bruta
        final = reco.sort_values('gross_yield', ascending=False).head(3)
    elif strategy == 'upside':
        # Ordenar por crecimiento histórico + infravaloración
        reco['score'] = reco['price_growth_1y'] - reco['desviacion_valor']
        final = reco.sort_values('score', ascending=False).head(3)
    else:
        # Ordenar por infravaloración pura (gangas)
        final = reco.sort_values('desviacion_valor', ascending=True).head(3)
        
    return final[['barrio_nombre', 'distrito_nombre', 'avg_venta_23', 'gross_yield', 'desviacion_valor', 'segmento']]

# EJEMPLO: Busco max rentabilidad con 200,000€
print("💡 SUGERENCIAS PARA INVERSIÓN DE 200k€ (Enfoque Cash-Flow/Yield):")
display(simulator(200000, strategy='yield'))

print("\n💡 SUGERENCIAS PARA INVERSIÓN DE 400k€ (Enfoque Oportunidad/Ganga):")
display(simulator(400000, strategy='safe'))

## 4. Conclusiones Finales <a name="4-conclusiones"></a>

1. **Eficiencia de Mercado**: Barcelona es un mercado maduro donde el precio está muy correlacionado con la renta familiar, pero el modelo de ML ha detectado desviaciones de hasta el 30% en barrios periféricos.
2. **Dualidad de Inversión**: Mientras que el centro (Eixample) ofrece seguridad patrimonial, la verdadera rentabilidad reside en barrios como **Nou Barris**, donde el yield supera el 6%.
3. **Próximos Pasos**: Integrar datos de criminalidad, equipamientos y proximidad al metro para refinar el modelo predictivo.